In [ ]:
# @title 6. Upload Specific Files for Validation
from google.colab import files

print("Upload ONE sheet image (PNG):")
uploaded_img = files.upload()
img_name = next(iter(uploaded_img))

print("\nUpload the matching vector JSON:")
uploaded_json = files.upload()
json_name = next(iter(uploaded_json))

print(f"\n✅ Ready to visualize: {img_name} with {json_name}")

In [ ]:
# @title 7. Multi-Layer Visualization
from PIL import Image, ImageDraw
import json
import matplotlib.pyplot as plt

# 1. Load image and JSON
img = Image.open(img_name).convert("RGB")
draw = ImageDraw.Draw(img)

with open(json_name, "r") as f:
    data = json.load(f)

# 2. Get dimensions for scaling
img_w, img_h = img.size
pdf_w = data["dims"]["w"]
pdf_h = data["dims"]["h"]

# 3. Compute scaling factors (PDF Points to Image Pixels)
x_scale = img_w / pdf_w
y_scale = img_h / pdf_h

print(f"Image size: {img_w} x {img_h} | PDF size: {pdf_w} x {pdf_h}")

# 4. Draw VECTOR TEXT (RED) - The original selectable text
for item in data.get("text", []):
    x0, y0, x1, y1 = item["bbox"]
    box = [x0 * x_scale, y0 * y_scale, x1 * x_scale, y1 * y_scale]
    draw.rectangle(box, outline="red", width=2)

# 5. Draw RASTER OCR (GREEN) - The sideways/missing text
for item in data.get("raster_ocr", []):
    # These are already in pixel coordinates, no scaling needed
    draw.rectangle(item["bbox"], outline="green", width=2)

# 6. Draw DETECTED LINES (BLUE) - The structural walls
for item in data.get("detected_lines", []):
    # These are already in pixel coordinates
    draw.line(item["start"] + item["end"], fill="blue", width=3)

# 7. Show result
plt.figure(figsize=(18, 24))
plt.imshow(img)
plt.axis("off")
plt.title(f"Validation: {img_name}\nRED=Vector Text | GREEN=Raster OCR | BLUE=Structural Lines", fontsize=16)
plt.show()

In [ ]:
# @title 8. High-Detail Zoomed View
from IPython.display import display

# This displays the image at its full pixel resolution in the Colab output
display(img)